# Cyber Attack Bayesian Network: Probability Generation

This notebook derives the conditional probability tables (CPTs) for the cyber-physical Bayesian network from the Hardware-in-the-Loop (HIL) Water Distribution Testbed dataset (Faramondi et al., *IEEE Access*, 2021).
Three sets of probabilities are computed:
- **Set 1** — P(AttackType_1): prior distribution over attack states for subsystem 1
- **Set 2** — P(AttackType_2 | AttackType_1): causal transition matrix between subsystems
- **Set 3** — P(Sensor | AttackType): sensor emission probabilities (data-derived, then expert-adjusted)

In [1]:
import pandas as pd
import numpy as np
import re

# ── Paths (edit to match your local data directory) ────────────────────
README_XLSX_PATH = "../cyber pipeline dataset/README.xlsx"
csv_paths     = [
    "../cyber pipeline dataset/Physical dataset/phy_att_1.csv",
    "../cyber pipeline dataset/Physical dataset/phy_att_2.csv",
    "../cyber pipeline dataset/Physical dataset/phy_att_3.csv",
    "../cyber pipeline dataset/Physical dataset/phy_att_4.csv",
]
readme_sheets = ["attack_1",      "attack_2",       "attack_3",       "attack_4"]

ATTACK_STATES = ["Normal", "MiTM", "DoS", "Physical"]
SENSOR_STATES = ["Normal", "Abnormal"]

# ── Subsystem asset mappings ────────────────────────────────────────────
# S1 = real subsystem, controlled by PLC1
# S2 = simulated subsystem interface, controlled by PLC2
#      (PLC3/T7 and PLC4/T8 are downstream simulation stages outside the
#       BN sensor scope and are excluded from the model)
S1 = {
    "tanks":  set(range(1, 6)),   # tanks 1-5
    "pumps":  set(range(1, 5)),   # pumps 1-4
    "valves": set(range(1, 20)),  # valves 1-19
    "flows":  set(),              # no flow sensor in S1
}
S2 = {
    "tanks":  {6},
    "pumps":  set(),
    "valves": {20},
    "flows":  {1},                # flow sensor 1
}
SUBSYSTEMS = {1: S1, 2: S2}

# ── Shared helpers (used by all three sets) ─────────────────────────────

def read_csv_robust(path):
    """Encoding-tolerant CSV reader; handles BOM and tab-separated files."""
    try:
        df = pd.read_csv(path)
        if df.shape[1] > 1:
            return df
    except Exception:
        pass
    for enc in ["utf-8-sig", "utf-8", "utf-16", "utf-16-le", "utf-16-be", "latin1", "cp1252"]:
        try:
            tmp = pd.read_csv(path, encoding=enc, nrows=5)
            if tmp.shape[1] == 1:
                header = tmp.columns[0]
                sep = "\t" if "\t" in header else (";" if ";" in header else None)
                if sep:
                    df = pd.read_csv(path, encoding=enc, sep=sep)
                    if df.shape[1] > 1:
                        return df
            df = pd.read_csv(path, encoding=enc)
            if df.shape[1] > 1:
                return df
        except (UnicodeDecodeError, Exception):
            continue
    raise RuntimeError(f"Could not read {path}")

def _find_col(df, candidates):
    """Return the first matching column name (case-insensitive partial match)."""
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    for c in df.columns:
        for cand in candidates:
            if cand.lower() in c.lower():
                return c
    raise KeyError(f"Could not find columns like {candidates}. Found: {list(df.columns)}")

def parse_attack_state(type_text):
    """Map README 'Type of attack' free text to one of the 4 canonical attack states."""
    s = str(type_text).lower()
    if "physical fault" in s or "breakdown" in s or "water leak" in s or "leak" in s:
        return "Physical"
    if "flood" in s or "dos" in s or "denial" in s or "icmp" in s or "tcp flood" in s or "udp flood" in s:
        return "DoS"
    if "mitm" in s or "man in the middle" in s or "scan" in s or "syn scan" in s or "fin scan" in s or "xmas" in s or "null scan" in s:
        return "MiTM"
    return "MiTM"  # conservative default

def extract_plcs(type_text):
    hits = re.findall(r"PLC\s*([1-4])", str(type_text), flags=re.IGNORECASE)
    return set(int(h) for h in hits)

def extract_assets(text):
    s = str(text).lower()
    tanks  = set(int(x) for x in re.findall(r"\btank\s*([0-9]+)\b", s))
    pumps  = set(int(x) for x in re.findall(r"\bpump\s*([0-9]+)\b", s))
    valves = set(int(x) for x in re.findall(r"\bvalve\s*([0-9]+)\b", s))
    flows  = set(int(x) for x in re.findall(r"\bflow\s*sensor\s*([0-9]+)\b", s))
    return tanks, pumps, valves, flows

def infer_subsystems(type_text, effect_phys):
    """Return the set of subsystem IDs (1 or 2) affected by an event row.

    Labelling rules:
    ─ Cyber attacks (MiTM / DoS): use only the PLC number in the attack-type
      text.  PLC1 → S1, PLC2 → S2.  PLC3 and PLC4 are downstream simulation
      stages (T7/T8) outside the BN sensor scope and are ignored.
      Effect text is NOT used — DoS on PLC1 causes a cascade delay in S2 but
      S2 is not itself under attack, so S2 should not be labelled DoS.

    ─ Physical attacks: use explicit asset-ID matching in the effect text.
      Physical faults are component-localised.
    """
    affected = set()
    attack_state = parse_attack_state(type_text)
    plcs = extract_plcs(type_text)

    if attack_state in ["MiTM", "DoS"]:
        if 1 in plcs: affected.add(1)
        if 2 in plcs: affected.add(2)
    else:  # Physical
        tanks, pumps, valves, flows = extract_assets(effect_phys)
        if (tanks & S1["tanks"]) or (pumps & S1["pumps"]) or (valves & S1["valves"]):
            affected.add(1)
        if (tanks & S2["tanks"]) or (pumps & S2["pumps"]) or (valves & S2["valves"]) or (flows & S2["flows"]):
            affected.add(2)
    return affected

def windows_from_readme_sheet(sheet_name):
    """Parse one README sheet; return a DataFrame of attack windows with subsystem assignments."""
    df = pd.ExcelFile(README_XLSX_PATH).parse(sheet_name)
    start_col = _find_col(df, ["Start time", "Start"])
    end_col   = _find_col(df, ["End time",   "End"])
    type_col  = _find_col(df, ["Type of attack", "Attack"])
    eff_col   = _find_col(df, ["Effect on physical process", "Effect on physical"])

    out = df[[start_col, end_col, type_col, eff_col]].copy()
    out.columns = ["start", "end", "type_text", "effect_phys"]
    out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
    out["end"]   = pd.to_datetime(out["end"],   errors="coerce", dayfirst=True)
    out = out.dropna(subset=["start", "end"])
    out = out[out["end"] >= out["start"]].copy()
    out["attack_state"] = out["type_text"].apply(parse_attack_state)
    out["subsystems"]   = out.apply(
        lambda r: infer_subsystems(r["type_text"], r["effect_phys"]), axis=1
    )
    return out[out["subsystems"].apply(len) > 0].copy()

### Set 1: P(AttackType_1) — Prior from data

Computed by labelling every sensor-data timestamp with its attack state (using the README attack windows), then counting across all four acquisitions.

In [2]:
def label_from_windows(times, windows_df, subsystem_id):
    labels = pd.Series(["Normal"] * len(times), index=times.index)
    w = windows_df[windows_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
    if w.empty:
        return labels
    for atk in ["MiTM", "DoS", "Physical"]:   # increasing precedence
        for _, row in w[w["attack_state"] == atk].iterrows():
            mask = (times >= row["start"]) & (times <= row["end"])
            labels.loc[mask] = atk
    return labels

def counts_from_labels(labels):
    c = labels.value_counts().to_dict()
    return {s: int(c.get(s, 0)) for s in ATTACK_STATES}

def add_counts(a, b):
    return {k: a.get(k, 0) + b.get(k, 0) for k in ATTACK_STATES}

def counts_to_prob(counts):
    p = np.array([counts[s] for s in ATTACK_STATES], dtype=float)
    total = p.sum()
    return p / total if total > 0 else p

def compute_attack_prior(subsystem_id=1):
    """Aggregate P(AttackType_subsystem) across all acquisitions."""
    total_counts = {s: 0 for s in ATTACK_STATES}
    per_acq = []

    for csv_path, sheet in zip(csv_paths, readme_sheets):
        df = read_csv_robust(csv_path)
        tcol = _find_col(df, ["Time", "Timestamp", "DateTime"])
        df[tcol] = pd.to_datetime(df[tcol], errors="coerce", dayfirst=True)
        df = df.dropna(subset=[tcol]).sort_values(tcol).reset_index(drop=True)
        times = df[tcol]
        t0, t1 = times.iloc[0], times.iloc[-1]

        win = windows_from_readme_sheet(sheet)
        win = win[(win["end"] >= t0) & (win["start"] <= t1)].copy()

        labels = label_from_windows(times, win, subsystem_id)
        c = counts_from_labels(labels)
        total_counts = add_counts(total_counts, c)
        per_acq.append((sheet, t0, t1, len(labels), c))

    p = counts_to_prob(total_counts)
    return p, total_counts, per_acq

# ── Compute ────────────────────────────────────────────────────────────
p_attack1_all, counts_s1_all, per_acq_s1 = compute_attack_prior(subsystem_id=1)

print("Total counts S1:", counts_s1_all)
print("P(AttackType_1) [Normal, MiTM, DoS, Physical]:", p_attack1_all.round(8))
print("\nPer acquisition:")
for sheet, t0, t1, n, c in per_acq_s1:
    print(f"  {sheet} | n={n} | {c}")

Total counts S1: {'Normal': 6574, 'MiTM': 91, 'DoS': 280, 'Physical': 550}
P(AttackType_1) [Normal, MiTM, DoS, Physical]: [0.87711808 0.01214143 0.03735824 0.07338225]

Per acquisition:
  attack_1 | n=2420 | {'Normal': 2055, 'MiTM': 88, 'DoS': 0, 'Physical': 277}
  attack_2 | n=2104 | {'Normal': 1980, 'MiTM': 2, 'DoS': 80, 'Physical': 42}
  attack_3 | n=1254 | {'Normal': 1109, 'MiTM': 0, 'DoS': 47, 'Physical': 98}
  attack_4 | n=1717 | {'Normal': 1430, 'MiTM': 1, 'DoS': 153, 'Physical': 133}


### Set 2: P(AttackType_2 | AttackType_1) — Transition matrix from data

Co-labels both subsystems at every timestep and counts joint occurrences, then applies Laplace smoothing (alpha=0.5) to avoid zero entries.

In [3]:
def build_attack_series(times, windows_df, subsystem_id):
    labels = pd.Series(["Normal"] * len(times), index=times.index)
    w = windows_df[windows_df["subsystems"].apply(lambda s: subsystem_id in s)].copy()
    if w.empty:
        return labels
    for atk in ["MiTM", "DoS", "Physical"]:
        for _, row in w[w["attack_state"] == atk].iterrows():
            mask = (times >= row["start"]) & (times <= row["end"])
            labels.loc[mask] = atk
    return labels

def smooth_and_normalize(counts, alpha=0.5):
    counts_sm = counts.astype(float) + alpha
    return counts_sm / counts_sm.sum(axis=0, keepdims=True)

# ── Compute ────────────────────────────────────────────────────────────
state_to_idx = {s: i for i, s in enumerate(ATTACK_STATES)}
counts_21_total = np.zeros((4, 4), dtype=int)
per_acq_s2 = []

for csv_path, sheet in zip(csv_paths, readme_sheets):
    df = read_csv_robust(csv_path)
    tcol = _find_col(df, ["Time", "Timestamp", "DateTime"])
    df[tcol] = pd.to_datetime(df[tcol], errors="coerce", dayfirst=True)
    df = df.dropna(subset=[tcol]).sort_values(tcol).reset_index(drop=True)
    times = df[tcol]
    t0, t1 = times.iloc[0], times.iloc[-1]

    win = windows_from_readme_sheet(sheet)
    win = win[(win["end"] >= t0) & (win["start"] <= t1)].copy()

    A1 = build_attack_series(times, win, subsystem_id=1)
    A2 = build_attack_series(times, win, subsystem_id=2)

    for c, p in zip(A2.values, A1.values):
        counts_21_total[state_to_idx[c], state_to_idx[p]] += 1

    per_acq_s2.append({
        "sheet": sheet, "n": len(times),
        "A1": A1.value_counts().to_dict(), "A2": A2.value_counts().to_dict()
    })

alpha = 0.5
cpt_21 = smooth_and_normalize(counts_21_total, alpha=alpha)

print("Raw joint counts [A2 x A1]:")
print(pd.DataFrame(counts_21_total, index=[f"A2={s}" for s in ATTACK_STATES],
                                    columns=[f"A1={s}" for s in ATTACK_STATES]))
print(f"\nCPT P(AttackType_2 | AttackType_1) with alpha={alpha}:")
print(pd.DataFrame(cpt_21, index=[f"A2={s}" for s in ATTACK_STATES],
                            columns=[f"A1={s}" for s in ATTACK_STATES]).round(4))
print("\nPer acquisition:")
for r in per_acq_s2:
    print(f"  {r['sheet']} | n={r['n']} | A1={r['A1']} | A2={r['A2']}")

Raw joint counts [A2 x A1]:
             A1=Normal  A1=MiTM  A1=DoS  A1=Physical
A2=Normal         6287        3     251          550
A2=MiTM            151       88       0            0
A2=DoS               0        0      29            0
A2=Physical        136        0       0            0

CPT P(AttackType_2 | AttackType_1) with alpha=0.5:
             A1=Normal  A1=MiTM  A1=DoS  A1=Physical
A2=Normal       0.9561   0.0376  0.8918       0.9973
A2=MiTM         0.0230   0.9516  0.0018       0.0009
A2=DoS          0.0001   0.0054  0.1046       0.0009
A2=Physical     0.0208   0.0054  0.0018       0.0009

Per acquisition:
  attack_1 | n=2420 | A1={'Normal': 2055, 'Physical': 277, 'MiTM': 88} | A2={'Normal': 2185, 'MiTM': 235}
  attack_2 | n=2104 | A1={'Normal': 1980, 'DoS': 80, 'Physical': 42, 'MiTM': 2} | A2={'Normal': 1993, 'Physical': 81, 'DoS': 29, 'MiTM': 1}
  attack_3 | n=1254 | A1={'Normal': 1109, 'Physical': 98, 'DoS': 47} | A2={'Normal': 1199, 'Physical': 55}
  attack_4 | n=1717

### Set 3: P(Sensor | AttackType) — Emission CPTs from data

For each sensor in each subsystem, timestamps are labelled with both the attack state and whether the sensor reading is Normal/Abnormal based on whether the event directly affects that sensor's assets. Laplace smoothing (alpha=0.5) is applied.

In [4]:
def load_attack_sheet(sheet_name):
    """Richer README parser that extracts PLC, tank, pump, valve, and flow IDs per row."""
    df = pd.read_excel(README_XLSX_PATH, sheet_name=sheet_name)

    def pick(patterns):
        for p in patterns:
            for c in df.columns:
                if p in c.lower():
                    return c
        raise KeyError(f"Could not find column matching {patterns}. Columns: {list(df.columns)}")

    start_col  = pick(["start"])
    end_col    = pick(["end"])
    type_col   = pick(["type of attack", "attack type", "type"])
    effect_col = pick(["effect on physical process", "effect on physical"])

    out = df[[start_col, end_col, type_col, effect_col]].copy()
    out.columns = ["start", "end", "type_text", "effect_phys"]
    out["start"] = pd.to_datetime(out["start"], errors="coerce", dayfirst=True)
    out["end"]   = pd.to_datetime(out["end"],   errors="coerce", dayfirst=True)
    out = out.dropna(subset=["start", "end"])
    out = out[out["end"] >= out["start"]].copy()
    out["attack_state"] = out["type_text"].apply(parse_attack_state)

    plcs_l, tanks_l, pumps_l, valves_l, flows_l = [], [], [], [], []
    for _, r in out.iterrows():
        plcs = extract_plcs(r["type_text"])
        t1, p1, v1, f1 = extract_assets(r["type_text"])
        t2, p2, v2, f2 = extract_assets(r["effect_phys"])
        plcs_l.append(plcs)
        tanks_l.append(t1 | t2);  pumps_l.append(p1 | p2)
        valves_l.append(v1 | v2); flows_l.append(f1 | f2)

    out["plcs"]   = plcs_l
    out["tanks"]  = tanks_l;  out["pumps"]  = pumps_l
    out["valves"] = valves_l; out["flows"]  = flows_l
    return out

def build_labels_for_sensor(times, events_df, subsystem_id, sensor_type):
    """
    Returns (attack_labels, sensor_labels) for the given sensor in the given subsystem.
    sensor_type in {'tank', 'pump', 'valve', 'flow'}.

    Labelling rules:
    ─ Cyber attacks (MiTM / DoS): the attack targets the subsystem PLC, so every
      sensor controlled by that PLC is affected.  Once sub_affected() confirms the
      subsystem is targeted (via PLC number in the attack-type text), sensor_affected
      returns True for every sensor that exists in the subsystem.

    ─ Physical attacks: faults are component-localised.  sensor_affected requires an
      explicit asset-ID match between the README effect text and the sensor's assets.
    """
    assets   = SUBSYSTEMS[subsystem_id]
    asset_key = sensor_type + "s" if sensor_type != "flow" else "flows"
    n = len(times)

    attack_labels = pd.Series(["Normal"] * n)
    sensor_labels = pd.Series(["Normal"] * n)

    if len(assets[asset_key]) == 0:
        return attack_labels, sensor_labels   # sensor doesn't exist in this subsystem

    csv_start, csv_end = times.iloc[0], times.iloc[-1]
    e = events_df[(events_df["start"] <= csv_end) & (events_df["end"] >= csv_start)].copy()
    if e.empty:
        return attack_labels, sensor_labels

    def sub_affected(row):
        if row["attack_state"] in ["MiTM", "DoS"]:
            return subsystem_id in row["plcs"]
        if row["attack_state"] == "Physical":
            return bool(
                (row["tanks"] & assets["tanks"]) or (row["pumps"] & assets["pumps"]) or
                (row["valves"] & assets["valves"]) or (row["flows"] & assets["flows"])
            )
        return False

    def sensor_affected(row):
        if row["attack_state"] in ["MiTM", "DoS"]:
            # Cyber attacks propagate through the PLC to ALL sensors in the subsystem.
            # sub_affected() already confirmed this subsystem's PLC is targeted; the
            # README text need not enumerate individual sensor IDs for this to hold.
            return len(assets[asset_key]) > 0   # True iff sensor type exists here
        # Physical fault: component-localised — require explicit asset-ID match
        return len(row[asset_key] & assets[asset_key]) > 0

    for atk in ["MiTM", "DoS", "Physical"]:
        for _, row in e[e["attack_state"] == atk].iterrows():
            if not sub_affected(row):
                continue
            mask = (times >= row["start"]) & (times <= row["end"])
            attack_labels.loc[mask] = atk
            if sensor_affected(row):
                sensor_labels.loc[mask] = "Abnormal"

    return attack_labels, sensor_labels

def compute_emission_cpt(subsystem_id, sensor_type, alpha=0.5):
    a2i = {s: i for i, s in enumerate(ATTACK_STATES)}
    counts = np.zeros((2, 4), dtype=int)

    for csv_path, sheet in zip(csv_paths, readme_sheets):
        df = read_csv_robust(csv_path)
        tcol = _find_col(df, ["Time", "Timestamp", "DateTime"])
        df[tcol] = pd.to_datetime(df[tcol], errors="coerce", dayfirst=True)
        df = df.dropna(subset=[tcol]).sort_values(tcol).reset_index(drop=True)
        times = df[tcol]

        events = load_attack_sheet(sheet)
        atk_lbl, sen_lbl = build_labels_for_sensor(times, events, subsystem_id, sensor_type)

        for s, a in zip(sen_lbl.values, atk_lbl.values):
            counts[0 if s == "Normal" else 1, a2i[a]] += 1

    counts_sm = counts.astype(float) + alpha
    cpt = counts_sm / counts_sm.sum(axis=0, keepdims=True)
    return counts, cpt

# ── Compute all emission CPTs ──────────────────────────────────────────
results = {}
for sub in [1, 2]:
    for sensor in ["tank", "valve", "pump", "flow"]:
        key = f"S{sub}_{sensor}"
        counts, cpt = compute_emission_cpt(sub, sensor)
        results[key] = {"counts_total": counts, "cpt": cpt}
        print(f"Computed {key}")

# ── Print corrected Set 3 summary ──────────────────────────────────────
print("\n=== Set 3: P(Sensor=Abnormal | AttackType) — data-derived (corrected labelling) ===")
print(f"{'Sensor':<14}", "  ".join(f"P(Abn|{a})" for a in ATTACK_STATES))
for key in ["S1_valve", "S1_pump", "S1_tank", "S2_valve", "S2_flow", "S2_tank"]:
    p_ab = results[key]["cpt"][1]   # row 1 = Abnormal
    print(f"{key:<14}", "  ".join(f"{v:.4f}      " for v in p_ab))

Computed S1_tank
Computed S1_valve
Computed S1_pump
Computed S1_flow
Computed S2_tank
Computed S2_valve
Computed S2_pump
Computed S2_flow

=== Set 3: P(Sensor=Abnormal | AttackType) — data-derived (corrected labelling) ===
Sensor         P(Abn|Normal)  P(Abn|MiTM)  P(Abn|DoS)  P(Abn|Physical)
S1_valve       0.0001        0.9946        0.9982        0.0009      
S1_pump        0.0001        0.9946        0.9982        0.5272      
S1_tank        0.0001        0.9946        0.9982        0.9991      
S2_valve       0.0001        0.9979        0.9833        0.9964      
S2_flow        0.0001        0.9979        0.9833        0.0036      
S2_tank        0.0001        0.9979        0.9833        0.9964      


In [5]:
# ── Apply Hybrid Set 3 CPT ────────────────────────────────────────────────────
# The data-derived `results` dict (from compute_emission_cpt above) is retained
# for transparency and raw-count inspection.  The final CPT values used in the BN
# are assembled here using the hybrid methodology (see markdown above).
#
# Array layout: shape (2, 4), rows = [Normal, Abnormal], cols = [Normal, MiTM, DoS, Physical]
#
# Column sources:
#   [0] Normal   — ISA-TR84 false-alert floor (0.01); replaces Laplace-only floor
#   [1] MiTM     — Rule-based stealth value (0.10); MITRE ATT&CK T0856/T0830
#   [2] DoS      — Rule-based PLC-starvation value (0.80); MITRE ATT&CK T0814
#   [3] Physical — Sensor-type tier: tank=0.80, valve=0.60, pump/flow=0.55
#                  Consistent across subsystems; data anchor = S1_pump 290/550≈0.527

P_NORMAL  = 0.01   # Normal column: ISA-TR84 instrument false-alert floor
P_MITM    = 0.10   # MiTM column:   stealthy attack — SCADA reads injected normal values
P_DOS     = 0.80   # DoS column:    PLC starvation → process drifts → sensor deviation

# Physical column tier values — by sensor type, applied identically across subsystems
PHYS_TANK  = 0.80  # integrating variable — accumulates any subsystem fault
PHYS_VALVE = 0.60  # actuator/fault-target — bidirectional role
PHYS_FLOW  = 0.55  # downstream process variable — secondary effect
PHYS_PUMP  = 0.55  # mechanical component — data anchor 290/550≈0.527

def make_hybrid_cpt(p_abn_normal, p_abn_mitm, p_abn_dos, p_abn_phys):
    """Build a (2,4) CPT from the four P(Abnormal|AttackType) scalars."""
    p_abn = np.array([p_abn_normal, p_abn_mitm, p_abn_dos, p_abn_phys])
    return np.vstack([1 - p_abn, p_abn])  # row0=Normal, row1=Abnormal

hybrid = {
    # Subsystem 1
    "S1_valve": make_hybrid_cpt(P_NORMAL, P_MITM, P_DOS, PHYS_VALVE),
    "S1_pump":  make_hybrid_cpt(P_NORMAL, P_MITM, P_DOS, PHYS_PUMP),
    "S1_tank":  make_hybrid_cpt(P_NORMAL, P_MITM, P_DOS, PHYS_TANK),
    # Subsystem 2  (no pump in S2)
    "S2_valve": make_hybrid_cpt(P_NORMAL, P_MITM, P_DOS, PHYS_VALVE),
    "S2_flow":  make_hybrid_cpt(P_NORMAL, P_MITM, P_DOS, PHYS_FLOW),
    "S2_tank":  make_hybrid_cpt(P_NORMAL, P_MITM, P_DOS, PHYS_TANK),
}

print("=== Set 3 — Final Hybrid CPT: P(Sensor=Abnormal | AttackType) ===")
print(f"{'Sensor':<12}  {'Normal':>8}  {'MiTM':>8}  {'DoS':>8}  {'Physical':>10}  {'Source: Physical'}")
print("-" * 80)
tier_label = {
    "S1_valve": f"valve tier ({PHYS_VALVE})",
    "S1_pump":  f"pump tier  ({PHYS_PUMP}) ← data anchor 290/550",
    "S1_tank":  f"tank tier  ({PHYS_TANK})",
    "S2_valve": f"valve tier ({PHYS_VALVE})",
    "S2_flow":  f"flow tier  ({PHYS_FLOW})",
    "S2_tank":  f"tank tier  ({PHYS_TANK})",
}
for key in ["S1_valve","S1_pump","S1_tank","S2_valve","S2_flow","S2_tank"]:
    p = hybrid[key][1]  # Abnormal row
    print(f"{key:<12}  {p[0]:>8.2f}  {p[1]:>8.2f}  {p[2]:>8.2f}  {p[3]:>10.2f}  {tier_label[key]}")
print()
print(f"Normal col:  ISA-TR84 false-alert floor ({P_NORMAL})")
print(f"MiTM col:    MITRE ATT&CK T0856/T0830 stealth property ({P_MITM})")
print(f"DoS col:     MITRE ATT&CK T0814 PLC starvation ({P_DOS})")


=== Set 3 — Final Hybrid CPT: P(Sensor=Abnormal | AttackType) ===
Sensor          Normal      MiTM       DoS    Physical  Source: Physical
--------------------------------------------------------------------------------
S1_valve          0.01      0.10      0.80        0.60  valve tier (0.6)
S1_pump           0.01      0.10      0.80        0.55  pump tier  (0.55) ← data anchor 290/550
S1_tank           0.01      0.10      0.80        0.80  tank tier  (0.8)
S2_valve          0.01      0.10      0.80        0.60  valve tier (0.6)
S2_flow           0.01      0.10      0.80        0.55  flow tier  (0.55)
S2_tank           0.01      0.10      0.80        0.80  tank tier  (0.8)

Normal col:  ISA-TR84 false-alert floor (0.01)
MiTM col:    MITRE ATT&CK T0856/T0830 stealth property (0.1)
DoS col:     MITRE ATT&CK T0814 PLC starvation (0.8)


In [6]:
k = "S2_pump"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S2_pump
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7495    0    0    0]
 [   0    0    0    0]]

CPT P(SensorState | AttackType):
          Normal  MiTM  DoS  Physical
Normal    0.9999   0.5  0.5       0.5
Abnormal  0.0001   0.5  0.5       0.5


In [7]:
k = "S2_valve"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S2_valve
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7091    0    0    0]
 [   0  239   29  136]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.0021  0.0167    0.0036
Abnormal  0.0001  0.9979  0.9833    0.9964


In [8]:
k = "S2_tank"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S2_tank
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7091    0    0    0]
 [   0  239   29  136]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.0021  0.0167    0.0036
Abnormal  0.0001  0.9979  0.9833    0.9964


In [9]:
k = "S1_valve"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_valve
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[6574    0    0  550]
 [   0   91  280    0]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.0054  0.0018    0.9991
Abnormal  0.0001  0.9946  0.9982    0.0009


In [10]:
k = "S1_tank"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_tank
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[6574    0    0    0]
 [   0   91  280  550]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.0054  0.0018    0.0009
Abnormal  0.0001  0.9946  0.9982    0.9991


In [11]:
k = "S1_pump"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_pump
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[6574    0    0  260]
 [   0   91  280  290]]

CPT P(SensorState | AttackType):
          Normal    MiTM     DoS  Physical
Normal    0.9999  0.0054  0.0018    0.4728
Abnormal  0.0001  0.9946  0.9982    0.5272


In [12]:
k = "S1_flow"
print("\n", k)
print("Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:\n", results[k]["counts_total"])
print("\nCPT P(SensorState | AttackType):")
print(pd.DataFrame(results[k]["cpt"], index=SENSOR_STATES, columns=ATTACK_STATES).round(4))


 S1_flow
Counts [Normal/Abnormal x AttackType (Normal,MiTM,DoS,Physical)]:
 [[7495    0    0    0]
 [   0    0    0    0]]

CPT P(SensorState | AttackType):
          Normal  MiTM  DoS  Physical
Normal    0.9999   0.5  0.5       0.5
Abnormal  0.0001   0.5  0.5       0.5


### Set 3: Hybrid CPT Methodology

Set 3 uses a **hybrid methodology**: the four attack-type columns are derived by
different means, chosen to match the physical and cyber mechanisms of each attack.

#### Column-by-column derivation

| Column | Method | Value | Justification |
|--------|--------|-------|---------------|
| **Normal** | ISA-TR84 standard floor | 0.01 | Dataset shows 0 false positives (testbed artefact); real deployments have calibration drift and measurement noise. ISA-TR84.00.09 specifies ~1 % false-alert rate for well-calibrated water-infrastructure instruments. |
| **MiTM** | Rule-based (MITRE ATT&CK for ICS) | 0.10 | MiTM attacks (ARP spoofing, false data injection) are designed to be stealthy: the attacker replays legitimate-looking values to the SCADA system. Sensor nodes in this BN represent SCADA-reported readings, so P(Abn\|MiTM) is low. Value = 0.10 rather than 0 because no spoofing is perfect — timing jitter and polling mismatches produce brief detectable inconsistencies. **Refs:** MITRE ATT&CK for ICS T0856 (Spoof Reporting Message), T0830 (Manipulate I/O Image); Liu et al. 2011 (stealthy false data injection). Applied uniformly across sensor types because stealth operates at PLC/network level. |
| **DoS** | Rule-based (MITRE ATT&CK for ICS) | 0.80 | DoS floods starve the PLC so it stops issuing control commands. The uncontrolled process drifts from setpoint and sensor readings become abnormal (stale values, fail-safe defaults, or out-of-range process variables). Not 1.0 because watchdog timers hold last output for a grace period and physical processes have inertia — brief DoS bursts may not produce detectable deviation within every timestep. **Refs:** MITRE ATT&CK for ICS T0814 (Denial of Service); Ralston et al. 2007 (ISA Transactions). Applied uniformly across sensor types for same reason as MiTM. |
| **Physical** | Sensor-type tier system (data-informed) | See below | Physical faults are component-localised; probabilities assigned by functional role. |

#### Physical column — sensor-type tier system

Physical faults (pump failures, valve faults, pipe leaks) affect sensors differently
depending on their functional role in the water distribution process.
Probabilities are defined **by sensor type**, applied consistently across both
subsystems — because the same hardware type plays the same physical role regardless
of which subsystem it is in.  Differences driven solely by the specific fault
scenarios present in the HIL dataset are **not** used to justify asymmetry: any
subsystem can experience any fault type in a real deployment.

| Sensor type | Role | P(Abn\|Physical) | Reasoning |
|-------------|------|-----------------|-----------|
| **Tank** | Integrating state variable | **0.80** | Tank level accumulates any mass-flow imbalance from upstream fault (pump failure, valve fault, pipe leak). Most subsystem faults eventually manifest here. Not 1.0 — some faults are isolated before level exceeds the anomaly threshold. Consistent with IEC 61511 diagnostic coverage for level sensors (0.90–0.99 for safety-rated SIS; lower for research testbeds). |
| **Valve** | Actuator and fault target | **0.60** | Valves are both potential fault targets (stuck actuator) and control responses to other faults. Bidirectional role gives moderate probability. IEC 61511 diagnostic coverage for valve position sensors: 0.60–0.90. Applied identically to S1 and S2 valves. |
| **Flow** | Downstream process variable | **0.55** | Flow changes when upstream valve or pump deviates; secondary/downstream effect. Same tier as pump. |
| **Pump** | Mechanical component | **0.55** | Data anchor: S1_pump raw count = 290/550 ≈ 0.527 (Laplace α=0.5). The 550 denominator spans all S1 physical fault types, diluting the pump signal; 0.55 preserves the empirical anchor rounded to a tier value. S2 has no pump. |

**Data-derived raw counts (for transparency):**
```
Sensor       Normal-window  Physical-window (Abn/Total)  Data P   Tier P
S1_valve     0/6574         0/550                        0.001    0.60
S1_pump      0/6574         290/550                      0.527    0.55  ← anchor
S1_tank      0/6574         550/550                      0.999    0.80
S2_valve     0/7091         136/136                      0.996    0.60
S2_flow      0/7091         0/136                        0.004    0.55
S2_tank      0/7091         136/136                      0.996    0.80
```
The near-deterministic data values (0.999, 0.996) are a labelling artefact: every
timestamp inside an attack window is stamped as Abnormal once the README names that
asset, regardless of whether the reading actually deviated. The tier values correct
for process inertia and detection lag.

### Validation: Classical Exact Inference (pgmpy)

The BN is reconstructed using `pgmpy` with the data-derived Sets 1 and 2 plus the expert-adjusted Set 3, then exact Variable Elimination is run to confirm the probabilities match expectations.

In [13]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

attack_order = ATTACK_STATES
binary_order = SENSOR_STATES

# Sets 1 and 2 from the computations above
P_A1 = p_attack1_all
A21  = cpt_21

# ── Set 3: hybrid CPT (assembled in cell above) ──────────────────────────
# `hybrid` dict produced by make_hybrid_cpt(); see hybrid_set3 cell for
# full methodology rationale and citation trail.
E_valve1 = hybrid["S1_valve"]
E_pump1  = hybrid["S1_pump"]
E_tank1  = hybrid["S1_tank"]
E_valve2 = hybrid["S2_valve"]
E_flow2  = hybrid["S2_flow"]
E_tank2  = hybrid["S2_tank"]

print("=== Set 3 — final P(Sensor=Abnormal | AttackType) [Normal, MiTM, DoS, Physical] ===")
for name, cpt in [("ValveState_1", E_valve1), ("PumpState_1",  E_pump1),
                  ("TankState_1",  E_tank1),  ("ValveState_2", E_valve2),
                  ("FlowState_2",  E_flow2),  ("TankState_2",  E_tank2)]:
    print(f"  {name:<14}: {np.round(cpt[1], 4)}")

model = DiscreteBayesianNetwork([
    ("AttackType_1", "AttackType_2"),
    ("AttackType_1", "ValveState_1"), ("AttackType_1", "PumpState_1"), ("AttackType_1", "TankState_1"),
    ("AttackType_2", "ValveState_2"), ("AttackType_2", "FlowState_2"), ("AttackType_2", "TankState_2"),
])

model.add_cpds(
    TabularCPD("AttackType_1", 4, P_A1.reshape(4, 1),
               state_names={"AttackType_1": attack_order}),
    TabularCPD("AttackType_2", 4, A21, evidence=["AttackType_1"], evidence_card=[4],
               state_names={"AttackType_2": attack_order, "AttackType_1": attack_order}),
    TabularCPD("ValveState_1", 2, E_valve1, evidence=["AttackType_1"], evidence_card=[4],
               state_names={"ValveState_1": binary_order, "AttackType_1": attack_order}),
    TabularCPD("PumpState_1",  2, E_pump1,  evidence=["AttackType_1"], evidence_card=[4],
               state_names={"PumpState_1":  binary_order, "AttackType_1": attack_order}),
    TabularCPD("TankState_1",  2, E_tank1,  evidence=["AttackType_1"], evidence_card=[4],
               state_names={"TankState_1":  binary_order, "AttackType_1": attack_order}),
    TabularCPD("ValveState_2", 2, E_valve2, evidence=["AttackType_2"], evidence_card=[4],
               state_names={"ValveState_2": binary_order, "AttackType_2": attack_order}),
    TabularCPD("FlowState_2",  2, E_flow2,  evidence=["AttackType_2"], evidence_card=[4],
               state_names={"FlowState_2":  binary_order, "AttackType_2": attack_order}),
    TabularCPD("TankState_2",  2, E_tank2,  evidence=["AttackType_2"], evidence_card=[4],
               state_names={"TankState_2":  binary_order, "AttackType_2": attack_order}),
)
assert model.check_model(), "BN is invalid — check CPTs."

infer = VariableElimination(model)

# Q1: Forward diagnosis — P(AttackType_2 | Flow2=Abnormal, Valve2=Abnormal)
evidence = {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"}
posterior = infer.query(["AttackType_2"], evidence=evidence, show_progress=False)
print("\nP(AttackType_2 | Flow2=Abnormal, Valve2=Abnormal):")
for s, v in zip(posterior.state_names["AttackType_2"], posterior.values):
    print(f"  {s}: {float(v):.6f}")

/Users/dengimoweialabintei/anaconda3/envs/qbn-pipeline/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Set 3 — final P(Sensor=Abnormal | AttackType) [Normal, MiTM, DoS, Physical] ===
  ValveState_1  : [0.01 0.1  0.8  0.6 ]
  PumpState_1   : [0.01 0.1  0.8  0.55]
  TankState_1   : [0.01 0.1  0.8  0.8 ]
  ValveState_2  : [0.01 0.1  0.8  0.6 ]
  FlowState_2   : [0.01 0.1  0.8  0.55]
  TankState_2   : [0.01 0.1  0.8  0.8 ]

P(AttackType_2 | Flow2=Abnormal, Valve2=Abnormal):
  Normal: 0.010374
  MiTM: 0.034990
  DoS: 0.288327
  Physical: 0.666309


In [14]:
# Q2: Upstream inference — P(AttackType_1 | Flow2=Abnormal, Valve2=Abnormal)
evidence = {"FlowState_2": "Abnormal", "ValveState_2": "Abnormal"}
posterior = infer.query(["AttackType_1"], evidence=evidence, show_progress=False)
print("P(AttackType_1 | Flow2=Abnormal, Valve2=Abnormal):")
for s, v in zip(posterior.state_names["AttackType_1"], posterior.values):
    print(f"  {s}: {float(v):.6f}")

P(AttackType_1 | Flow2=Abnormal, Valve2=Abnormal):
  Normal: 0.695193
  MiTM: 0.019627
  DoS: 0.277230
  Physical: 0.007949


In [15]:
# Q3: Conditional upstream — P(AttackType_1 | AttackType_2=MiTM)
evidence = {"AttackType_2": "MiTM"}
posterior = infer.query(["AttackType_1"], evidence=evidence, show_progress=False)
print("P(AttackType_1 | AttackType_2=MiTM):")
for s, v in zip(posterior.state_names["AttackType_1"], posterior.values):
    print(f"  {s}: {float(v):.6f}")

P(AttackType_1 | AttackType_2=MiTM):
  Normal: 0.633578
  MiTM: 0.362261
  DoS: 0.002077
  Physical: 0.002084


In [16]:
# ── Print exact constants for cyber_attack_bn.py ──────────────────────────────
# Copy-paste the block below into cyber_attack_bn.py to keep the two files in sync.
# Values come from the hybrid Set 3 CPT (see hybrid_set3 cell for full rationale).
def fmt2(arr):
    return "[" + ", ".join(f"{v:.2f}" for v in arr) + "]"

sensor_map = {
    "P_VALVE_S1": hybrid["S1_valve"][1],
    "P_PUMP_S1":  hybrid["S1_pump"][1],
    "P_TANK_S1":  hybrid["S1_tank"][1],
    "P_VALVE_S2": hybrid["S2_valve"][1],
    "P_FLOW_S2":  hybrid["S2_flow"][1],
    "P_TANK_S2":  hybrid["S2_tank"][1],
}

print("# ── Set 3: P(Sensor=Abnormal | AttackType)  [Normal, MiTM, DoS, Physical] ──────")
print("# Hybrid methodology: Normal=ISA-TR84 floor, MiTM=MITRE T0856/T0830 stealth,")
print("#   DoS=MITRE T0814 PLC starvation, Physical=sensor-type tier system.")
print("# See cyber_bn_cpt_generation.ipynb (hybrid_set3 cell) for full rationale.")
for name, vals in sensor_map.items():
    print(f"{name} = np.array({fmt2(vals)})")


# ── Set 3: P(Sensor=Abnormal | AttackType)  [Normal, MiTM, DoS, Physical] ──────
# Hybrid methodology: Normal=ISA-TR84 floor, MiTM=MITRE T0856/T0830 stealth,
#   DoS=MITRE T0814 PLC starvation, Physical=sensor-type tier system.
# See cyber_bn_cpt_generation.ipynb (hybrid_set3 cell) for full rationale.
P_VALVE_S1 = np.array([0.01, 0.10, 0.80, 0.60])
P_PUMP_S1 = np.array([0.01, 0.10, 0.80, 0.55])
P_TANK_S1 = np.array([0.01, 0.10, 0.80, 0.80])
P_VALVE_S2 = np.array([0.01, 0.10, 0.80, 0.60])
P_FLOW_S2 = np.array([0.01, 0.10, 0.80, 0.55])
P_TANK_S2 = np.array([0.01, 0.10, 0.80, 0.80])
